In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
%%writefile strands_claude.py

from strands import Agent, tool
from strands_tools import calculator
import argparse
import json
from strands.models import BedrockModel

@tool
def weather():
    return "very hot and UV levels are around 11"

model_id = "us.anthropic.claude-3-5-haiku-20241022-v1:0"

model = BedrockModel(
    model_id = model_id
)

agent = Agent(
    model = model,
    tools = [calculator,weather],
    system_prompt= "You are a helpful assistant. You can help with math calculations, and tell the weather"
)

def strands_agent_bedrock(payload):
    user_input = payload.get("prompt")
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("payload",type = str)
    args = parser.parse_args()
    response = strands_agent_bedrock(json.loads(args.payload))
    
    



In [ ]:
!python strands_claude.py '{"prompt": "what is the weather like? do i need a jacket today?"}'

In [ ]:
%%writefile strands_claude.py

from strands import Agent, tool
from strands_tools import calculator
import argparse
import json
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp

app = BedrockAgentCoreApp()

@tool
def weather():
    return "very hot and UV levels are around 11"

model_id = "us.anthropic.claude-3-5-haiku-20241022-v1:0"

model = BedrockModel(
    model_id = model_id
)

agent = Agent(
    model = model,
    tools = [calculator,weather],
    system_prompt= "You are a helpful assistant. You can help with math calculations, and tell the weather"
)

@app.entrypoint
def strands_agent_bedrock(payload):
    user_input = payload.get("prompt")
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()
    



In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "agent_demo"
response = agentcore_runtime.configure(
    entrypoint="strands_claude.py",
    auto_create_execution_role = True,
    auto_create_ecr=True,
    requirements_file="../../../../../Downloads/agentcore-main/AgentCoreRuntime/requirements.txt",
    region = region,
    agent_name = agent_name
)




In [ ]:
launch_result = agentcore_runtime.launch()

In [ ]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']

end_status = ['READY','CREATE_FAILED','DELETE_FAILED','UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response =  agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status
    

In [ ]:
import sagemaker

role = sagemaker.get_execution_role()
print(role)

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt":"what is 2+2"})
invoke_response

In [ ]:
from IPython.display import Markdown, display
import json
response_text = invoke_response['response'][0]
display(Markdown(response_text))

In [ ]:
!pip install --upgrade boto3 botocore


In [ ]:
import boto3
import json

client = boto3.client('bedrock-agentcore', region_name='us-west-2')
payload = json.dumps({"prompt": "what questions have I asked you recently"})

response = client.invoke_agent_runtime(
    agentRuntimeArn='arn:aws:bedrock-agentcore:us-west-2:862851468070:runtime/agent_demo-uvhgvuH6GV',
    runtimeSessionId='aoooooaksjfkasjfkklasjfklasjfklasjdfkl', # Must be 33+ char. Every new SessionId will create a new MicroVM
    payload=payload,
    qualifier="DEFAULT" # This is Optional. When the field is not provided, Runtime will use DEFAULT endpoint
)
response_body = response['response'].read()
response_data = json.loads(response_body)
print("Agent Response:", response_data)

In [ ]:
import boto3
import json

client = boto3.client('bedrock-agentcore', region_name='us-west-2')
payload = json.dumps({"prompt": "what is 66*6"})

response = client.invoke_agent_runtime(
    agentRuntimeArn='arn:aws:bedrock-agentcore:us-west-2:862851468070:runtime/agent_demo-uvhgvuH6GV',
    runtimeSessionId='aoasjfkasjfklaksjfklasjfklasjdfkl', # Must be 33+ char. Every new SessionId will create a new MicroVM
    payload=payload,
    qualifier="DEFAULT" # This is Optional. When the field is not provided, Runtime will use DEFAULT endpoint
)
response_body = response['response'].read()
response_data = json.loads(response_body)
print("Agent Response:", response_data)
print(response)
runtime_session_id = response['runtimeSessionId']
print(runtime_session_id)

In [ ]:
runtime_session_id

In [ ]:
if runtime_session_id:
    client.stop_runtime_session(
        agentRuntimeArn='arn:aws:bedrock-agentcore:us-west-2:862851468070:runtime/agent_demo-uvhgvuH6GV',
        runtimeSessionId = runtime_session_id,
        qualifier = 'DEFAULT'
    )

In [ ]:
payload = json.dumps({"prompt": "what calculation did you just do?"})
response = client.invoke_agent_runtime(
    agentRuntimeArn='arn:aws:bedrock-agentcore:us-west-2:862851468070:runtime/agent_demo-uvhgvuH6GV',
    runtimeSessionId='aoasjfkasjfklaksjfklasjfklasjdfkl', # Must be 33+ char. Every new SessionId will create a new MicroVM
    payload=payload,
    qualifier="DEFAULT" # This is Optional. When the field is not provided, Runtime will use DEFAULT endpoint
)
response_body = response['response'].read()
response_data = json.loads(response_body)
print("Agent Response:", response_data)

In [ ]:
agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name='us-west-2')

current = agentcore_control_client.get_agent_runtime(
    agentRuntimeId='agent_demo-uvhgvuH6GV'
)
print(current)

In [ ]:
agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name='us-west-2')
update_response = agentcore_control_client.update_agent_runtime(
    agentRuntimeId='agent_demo-uvhgvuH6GV',
    agentRuntimeArtifact={
        'containerConfiguration': {
            'containerUri': '862851468070.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-agent_demo:latest'
        }
    },
    roleArn='arn:aws:iam::862851468070:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-5435ab0047',
    networkConfiguration={'networkMode': 'PUBLIC'},
    lifecycleConfiguration={
        'idleRuntimeSessionTimeout': 180,  # 3 minutes
        'maxLifetime': 28800 # 8 hours
    }
)
print("Runtime updated with 3-minute idle timeout")